In [0]:
from pyspark.sql import functions as F, Window

# Update this to your real path
LCD_PATH = "/Volumes/workspace/default/databricks-hackathon/lcd/*.csv"

# Output tables
RAW_ENRICHED_TABLE = "workspace.default.noaa_lcd_raw_enriched"
CLEAN_TABLE = "workspace.default.noaa_lcd_clean"

MONTHLY_TABLE = "workspace.default.noaa_station_month_metrics"
YEARLY_TABLE = "workspace.default.noaa_station_year_metrics"

In [0]:
lcd_raw_enriched = (
    spark.read
    .option("header", True)
    .option("multiLine", False)
    .csv(LCD_PATH)
    .withColumn("source_file", F.col('_metadata.file_path'))
    .withColumn("file_name", F.regexp_extract("source_file", r"([^/]+$)", 1))
    .withColumn("state", F.regexp_extract("file_name", r"^([A-Z]{2})_", 1))
    .withColumn("station_id", F.regexp_extract("file_name", r"(USW\d+)", 1))
    .withColumn("file_year", F.regexp_extract("file_name", r"_(\d{4})\.csv$", 1).cast("int"))
    .withColumn(
        "station_name",
        F.regexp_extract("file_name", r"^[A-Z]{2}_(.+)_(USW\d+)_(\d{4})\.csv$", 1)
    )
)
# Remove all case variants before save
for colname in ["latitude", "LATITUDE", "longitude", "LONGITUDE", "elevation", "ELEVATION"]:
    if colname in lcd_raw_enriched.columns:
        lcd_raw_enriched = lcd_raw_enriched.drop(colname)
lcd_raw_enriched.write.mode("overwrite").saveAsTable(RAW_ENRICHED_TABLE)

display(lcd_raw_enriched.select("file_name", "state", "station_name", "station_id", "file_year").limit(20))
print(lcd_raw_enriched.columns)

In [0]:
from pyspark.sql import functions as F, Window

raw = spark.table(RAW_ENRICHED_TABLE)

def to_double_from_messy(colname):
    return F.regexp_extract(F.col(colname).cast("string"), r"(-?\d+(?:\.\d+)?)", 1).cast("double")

lcd_clean = (
    raw
    .withColumn("obs_time", F.to_timestamp("DATE"))
    .withColumn("date", F.to_date("obs_time"))
    .withColumn("year", F.year("obs_time"))
    .withColumn("month", F.month("obs_time"))
    .withColumn("hour", F.hour("obs_time"))
    .withColumn("temp_c", to_double_from_messy("HourlyDryBulbTemperature"))
    .withColumn("rh_pct", to_double_from_messy("HourlyRelativeHumidity"))
    .withColumn(
        "precip_mm",
        F.when(F.trim(F.col("HourlyPrecipitation").cast("string")) == "T", F.lit(0.0))
         .otherwise(to_double_from_messy("HourlyPrecipitation"))
    )
    .withColumn(
        "day_night",
        F.when((F.col("hour") >= 7) & (F.col("hour") < 19), F.lit("day"))
         .otherwise(F.lit("night"))
    )
    .filter(F.col("obs_time").isNotNull())
    .filter(F.col("year").between(2010, 2024))
    .select(
        "state",
        "station_name",
        "station_id",
        "file_year",
        "STATION",
        "NAME",
        F.col("LATITUDE").cast("double").alias("latitude"),
        F.col("LONGITUDE").cast("double").alias("longitude"),
        F.col("ELEVATION").cast("double").alias("elevation"),
        "obs_time",
        "date",
        "year",
        "month",
        "hour",
        "day_night",
        "temp_c",
        "rh_pct",
        "precip_mm"
    )
)

lcd_clean.write.mode("overwrite").saveAsTable(CLEAN_TABLE)
display(lcd_clean.limit(20))

In [0]:
from pyspark.sql import functions as F, Window

raw = spark.table(RAW_ENRICHED_TABLE)

def to_double_from_messy(colname):
    return F.regexp_extract(F.col(colname).cast("string"), r"(-?\d+(?:\.\d+)?)", 1).cast("double")

lcd_clean = (
    raw
    .withColumn("obs_time", F.to_timestamp("DATE"))
    .withColumn("date", F.to_date("obs_time"))
    .withColumn("year", F.year("obs_time"))
    .withColumn("month", F.month("obs_time"))
    .withColumn("hour", F.hour("obs_time"))
    .withColumn("temp_c", to_double_from_messy("HourlyDryBulbTemperature"))
    .withColumn("rh_pct", to_double_from_messy("HourlyRelativeHumidity"))
    .withColumn(
        "precip_mm",
        F.when(F.trim(F.col("HourlyPrecipitation").cast("string")) == "T", F.lit(0.0))
         .otherwise(to_double_from_messy("HourlyPrecipitation"))
    )
    .withColumn(
        "day_night",
        F.when((F.col("hour") >= 7) & (F.col("hour") < 19), F.lit("day"))
         .otherwise(F.lit("night"))
    )
    .filter(F.col("obs_time").isNotNull())
    .filter(F.col("year").between(2010, 2024))
    .select(
        "state",
        "station_name",
        "station_id",
        "file_year",
        "STATION",
        "NAME",
        F.col("LATITUDE").cast("double").alias("latitude"),
        F.col("LONGITUDE").cast("double").alias("longitude"),
        F.col("ELEVATION").cast("double").alias("elevation"),
        "obs_time",
        "date",
        "year",
        "month",
        "hour",
        "day_night",
        "temp_c",
        "rh_pct",
        "precip_mm"
    )
)

lcd_clean.write.mode("overwrite").saveAsTable(CLEAN_TABLE)
display(lcd_clean.limit(20))

In [0]:
lcd = spark.table(CLEAN_TABLE)

daily_station = (
    lcd
    .groupBy("state", "station_name", "station_id", "year", "month", "date")
    .agg(
        F.min("temp_c").alias("daily_min_temp_c"),
        F.avg("temp_c").alias("daily_avg_temp_c"),
        F.avg("rh_pct").alias("daily_avg_rh_pct"),
        F.sum(F.coalesce("precip_mm", F.lit(0.0))).alias("daily_precip_mm")
    )
    .withColumn("frost_free_day", F.col("daily_min_temp_c") > F.lit(0.0))
)

display(daily_station.limit(20))

In [0]:
station_years = (
    daily_station
    .select("state", "station_name", "station_id", "year")
    .distinct()
)

calendar = (
    station_years
    .withColumn("start_date", F.to_date(F.concat_ws("-", F.col("year"), F.lit("01"), F.lit("01"))))
    .withColumn("end_date", F.to_date(F.concat_ws("-", F.col("year"), F.lit("12"), F.lit("31"))))
    .withColumn("date", F.explode(F.sequence(F.col("start_date"), F.col("end_date"))))
    .drop("start_date", "end_date")
)

daily_full = (
    calendar.alias("c")
    .join(
        daily_station.alias("d"),
        on=[
            F.col("c.state") == F.col("d.state"),
            F.col("c.station_name") == F.col("d.station_name"),
            F.col("c.station_id") == F.col("d.station_id"),
            F.col("c.year") == F.col("d.year"),
            F.col("c.date") == F.col("d.date"),
        ],
        how="left"
    )
    .select(
        F.col("c.state").alias("state"),
        F.col("c.station_name").alias("station_name"),
        F.col("c.station_id").alias("station_id"),
        F.col("c.year").alias("year"),
        F.col("c.date").alias("date"),
        F.month(F.col("c.date")).alias("month"),
        F.coalesce(F.col("d.frost_free_day"), F.lit(False)).alias("frost_free_day")
    )
)

w = Window.partitionBy("state", "station_name", "station_id", "year").orderBy("date")

streak_base = (
    daily_full
    .withColumn("break_flag", F.when(F.col("frost_free_day"), F.lit(0)).otherwise(F.lit(1)))
    .withColumn("grp", F.sum("break_flag").over(w))
)

streak_lengths = (
    streak_base
    .filter(F.col("frost_free_day") == True)
    .groupBy("state", "station_name", "station_id", "year", "grp")
    .agg(F.count("*").alias("streak_days"))
)

frost_yearly = (
    streak_lengths
    .groupBy("state", "station_name", "station_id", "year")
    .agg(F.max("streak_days").alias("longest_frost_free_days"))
)

frost_yearly_full = (
    station_years
    .join(frost_yearly, on=["state", "station_name", "station_id", "year"], how="left")
    .withColumn("longest_frost_free_days", F.coalesce(F.col("longest_frost_free_days"), F.lit(0)))
    .withColumn("corn_frost_free_ok", F.col("longest_frost_free_days") >= 120)
)

display(frost_yearly_full.limit(20))

In [0]:
lcd = spark.table(CLEAN_TABLE)

monthly_weather = (
    lcd
    .groupBy("state", "station_name", "station_id", "year", "month")
    .agg(
        F.max("latitude").alias("latitude"),
        F.max("longitude").alias("longitude"),
        F.max("elevation").alias("elevation"),
        F.avg(F.when(F.col("day_night") == "day", F.col("temp_c"))).alias("day_temperature_c"),
        F.avg(F.when(F.col("day_night") == "night", F.col("temp_c"))).alias("night_temperature_c"),
        F.avg("rh_pct").alias("relative_humidity_pct"),
        F.sum(F.coalesce("precip_mm", F.lit(0.0))).alias("monthly_rainfall_mm")
    )
)

monthly_metrics = (
    monthly_weather.alias("m")
    .join(
        frost_yearly_full.alias("f"),
        on=["state", "station_name", "station_id", "year"],
        how="left"
    )
    .select(
        "state",
        "station_name",
        "station_id",
        F.round("latitude", 6).alias("latitude"),
        F.round("longitude", 6).alias("longitude"),
        F.round("elevation", 2).alias("elevation"),
        "year",
        "month",
        "day_temperature_c",
        "night_temperature_c",
        "relative_humidity_pct",
        "monthly_rainfall_mm",
        "longest_frost_free_days",
        "corn_frost_free_ok"
    )
)

monthly_metrics.write.mode("overwrite").saveAsTable(MONTHLY_TABLE)
display(monthly_metrics.orderBy("state", "station_name", "year", "month"))

In [0]:
seasonal_rain_yearly = (
    lcd
    .withColumn(
        "season",
        F.when(F.col("month").isin(12, 1, 2), "winter")
         .when(F.col("month").isin(3, 4, 5), "spring")
         .when(F.col("month").isin(6, 7, 8), "summer")
         .otherwise("fall")
    )
    .groupBy("state", "station_name", "station_id", "year")
    .pivot("season", ["winter", "spring", "summer", "fall"])
    .agg(F.sum(F.coalesce("precip_mm", F.lit(0.0))))
    .withColumnRenamed("winter", "winter_rainfall_mm")
    .withColumnRenamed("spring", "spring_rainfall_mm")
    .withColumnRenamed("summer", "summer_rainfall_mm")
    .withColumnRenamed("fall", "fall_rainfall_mm")
)

In [0]:
yearly_weather = (
    lcd
    .groupBy("state", "station_name", "station_id", "year")
    .agg(
        F.max("NAME").alias("station_long_name"),
        F.max("latitude").alias("latitude"),
        F.max("longitude").alias("longitude"),
        F.max("elevation").alias("elevation"),
        F.avg(F.when(F.col("day_night") == "day", F.col("temp_c"))).alias("day_temperature_c"),
        F.avg(F.when(F.col("day_night") == "night", F.col("temp_c"))).alias("night_temperature_c"),
        F.avg("rh_pct").alias("relative_humidity_pct")
    )
)

yearly_metrics = (
    yearly_weather.alias("y")
    .join(
        seasonal_rain_yearly.alias("s"),
        on=["state", "station_name", "station_id", "year"],
        how="left"
    )
    .join(
        frost_yearly_full.alias("f"),
        on=["state", "station_name", "station_id", "year"],
        how="left"
    )
    .select(
        "state",
        "station_name",
        "station_id",
        "station_long_name",
        F.round("latitude", 6).alias("latitude"),
        F.round("longitude", 6).alias("longitude"),
        F.round("elevation", 2).alias("elevation"),
        "year",
        F.round("day_temperature_c", 2).alias("day_temperature_c"),
        F.round("night_temperature_c", 2).alias("night_temperature_c"),
        F.round("relative_humidity_pct", 2).alias("relative_humidity_pct"),
        F.round(F.coalesce(F.col("winter_rainfall_mm"), F.lit(0.0)), 2).alias("winter_rainfall_mm"),
        F.round(F.coalesce(F.col("spring_rainfall_mm"), F.lit(0.0)), 2).alias("spring_rainfall_mm"),
        F.round(F.coalesce(F.col("summer_rainfall_mm"), F.lit(0.0)), 2).alias("summer_rainfall_mm"),
        F.round(F.coalesce(F.col("fall_rainfall_mm"), F.lit(0.0)), 2).alias("fall_rainfall_mm"),
        "longest_frost_free_days",
        "corn_frost_free_ok"
    )
)

yearly_metrics.write.mode("overwrite").saveAsTable(YEARLY_TABLE)
display(yearly_metrics.orderBy("state", "station_name", "year"))

In [0]:
monthly_metrics.write.mode("overwrite").saveAsTable(MONTHLY_TABLE)
yearly_metrics.write.mode("overwrite").saveAsTable(YEARLY_TABLE)

print(f"Saved monthly table: {MONTHLY_TABLE}")
print(f"Saved yearly table:  {YEARLY_TABLE}")